## Step by Step wastewater from shedding hub -> Covasim

# Build the EpiSewer input from Covasim (shedding_hub curves already checked)

**Goal:** make everything `EpiSewer(data = ww_data, assumptions = ww_assumptions, shedding = model_shedding(load_per_case = ...))` needs.
Units: shedding is gc/mL of stool (shedding_hub label, reference event symptom onset: already checked); the load is gc/day; the concentration goes to EpiSewer in gc/mL; the flow in mL/day.

## Settings
Epidemic: `N_AGENTS=50000`, `N_DAYS=180`, `TRANSMISSION_PROB_PER_CONTACT=0.012`, `N_INITIAL_INFECTIONS=10`, `FRIEND_COUNT_DISPERSION=None`, `INFECTIOUSNESS_DISPERSION=0.45`, `DAILY_TEST_PROB_IF_SYMPTOMATIC=0.1` (not used by EpiSewer; kept so the run matches run_covasim.py).
Maps for `pop_type='rank'`: `POPULATION_DENSITY_RASTER`, `ZIP_CODE_BOUNDARIES`, `SEWERSHED_BOUNDARIES` (same three files as run_covasim.py).

## Our assumptions
- `STOOL_ML_PER_PERSON_PER_DAY=128`: median fecal wet mass of 128 g per person per day, with 1 g taken as 1 mL [1]
- `SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY=270`: Point Loma treats 136 to 175 million gallons per day for about 2.2 million residents [2].
  - Calculation: 1 million gallons = 3.785 million L, so 136 MGD = 5.1 × 10⁸ L and 175 MGD = 6.6 × 10⁸ L per day.
  - Divided by 2.2 million residents this is 234 to 301 L per person per day. We use 270, the midpoint, rounded.
  - It is a per-resident average: the plant flow also includes commercial, industrial and infiltration water.
- `LAB_NOISE_CV=0.2`: placeholder lab noise (multiplicative, mean 1); no reference

## Technical settings
- `SHEDDING_PROFILE_DAYS=40`: days after infection kept when building the shedding distribution (check that the window holds about 100% of the shedding)
- A sample is taken every day (the best case)

## Steps
1. **Run Covasim** with the settings above. Copy `sim.shedding_lookup` after `initialize()` and before `run()`. Take `viral_shedding_hist` (agents × calendar days, gc/mL), `date_exposed`, `date_infectious` and `people.sewershed`.
2. **Check:** sliding each agent's curve (`shedding_lookup` row) to start at `date_infectious` reproduces `viral_shedding_hist`.
3. **Site load:** add up the agents (gc/mL), then × `STOOL_ML_PER_PERSON_PER_DAY` (128 mL), giving `load_gc_per_day`.
4. **Flow and concentration:** flow = agents × `SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY` (270 L). Concentration = load ÷ flow, in gc/L.
5. **Measurement:** concentration × lab noise (mean 1, CV 0.2). Leave out days with a concentration of 0.
6. **EpiSewer data tables:** concentration in gc/mL (÷ 1000) and flow in mL/day (× 1000). These are `ww_data`.
7. **Shedding distribution:** line each agent's shedding up by days since `date_exposed`, average over agents, and divide by the total (sums to 1). This is `shedding_dist`.
8. **Load per case:** mean total shedding of one infection × 128 mL, in gc per infection. This goes to `load_per_case_assume`.
9. **Generation time:** from `infection_log`, the infectee's infection day minus the infector's, from day 1. This is `generation_dist`.
10. **Save the CSVs** for the R script. For judging only: true infections per day from `infection_log`.

## What each EpiSewer argument comes from
| EpiSewer argument | From step |
|---|---|
| `data` (`ww_data`: concentration, flow) | 3–6 |
| `shedding_dist`, `shedding_reference = "infection"` | 7 |
| `load_per_case` | 8 |
| `generation_dist` | 9 |
| not given to EpiSewer: true infections | 10 |

## References
1. Rose C, Parker A, Jefferson B, Cartmell E. The characterization of feces and urine: a review of the literature to inform advanced treatment technology. *Crit Rev Environ Sci Technol* 2015; 45:1827–1879. doi:10.1080/10643389.2014.1000761
2. City of San Diego. Point Loma Wastewater Treatment Plant fact sheet (about 175 MGD, more than 2.2 million residents); Point Loma annual reports 2014 (139.2 MGD) and 2016 (136.1 MGD).

In [1]:
import covasim as cv

Covasim 3.1.8 (2026-05-29) — © 2020-2026 by IDM


In [ ]:
N_AGENTS = 5000
N_DAYS = 90

sim = cv.Sim(
    pop_type='random',        # the exact run_covasim.py setup uses 'rank' + the three map files
    pop_size=N_AGENTS,
    pop_infected=50,
    n_imports=0,
    n_days=N_DAYS,
    rand_seed=1,
    verbose=0,
)
sim.initialize()                       # this is where compute_viral_shedding_lookup() runs
lookup = sim.shedding_lookup.copy()    # copy NOW: Covasim redraws it after the run
sim.run()
hist = np.asarray(sim.results['viral_shedding_hist'], dtype=float)

print("lookup :", lookup.shape, "(agents, days since symptom onset, gc/mL)")
print("hist   :", hist.shape, "(agents, calendar days, gc/mL)")

### Diagram

## Flow: from shedding_hub output to the EpiSewer inputs

```
┌─ COVASIM OUTPUT (shedding_hub) ──────────────────────────────────────┐
│ viral_shedding_hist  (agents x days)                                 │
│ unit: gene copies per mL of STOOL (gc/mL)                            │
└──────────────────────────────────────────────────────────────────────┘
                  │                                    │
                  ▼                                    ▼
┌─ our arithmetic ─────────────────┐  ┌─ ASSUMPTION: 270 L ────────────┐
│ 3. add up agents in the site     │  │ 4a. flow = agents x 270 L      │
│    (still gc/mL)                 │  │     = flow_L_per_day (L/day)   │
└──────────────────────────────────┘  └────────────────────────────────┘
                  │                                    │
                  ▼                                    │
┌─ ASSUMPTION: 128 mL ─────────────┐                   │
│ 3b. x 128 mL of stool per day    │                   │
│     = load_gc_per_day (gc/day)   │                   │
└──────────────────────────────────┘                   │
                  │                                    │
                  └─────────────────┬──────────────────┘
                                    ▼
                   ┌─ our arithmetic ─────────────────┐
                   │ 4b. load / flow = concentration  │
                   │     (gc/L, no noise)             │
                   └──────────────────────────────────┘
                                    │
                                    ▼
                   ┌─ ASSUMPTION: noise ──────────────┐
                   │ 5. x lab noise (mean 1, CV 0.2)  │
                   │    drop days where it is 0       │
                   └──────────────────────────────────┘
                                    │
                                    ▼
                   ┌─ unit conversion ────────────────┐
                   │ 6. concentration / 1000 (gc/mL)  │
                   │    flow x 1000 (mL/day)          │
                   └──────────────────────────────────┘
                                    │
                                    ▼
                   ┌─ INPUT TO EPISEWER ──────────────┐
                   │ ww_data = concentration + flow   │
                   │ (gc/mL and mL/day)               │
                   └──────────────────────────────────┘

Also from the same Covasim tables (steps 7 to 9):
┌─ INPUT: shedding_dist ─────────┐ ┌─ INPUT: load_per_case ─────────┐ ┌─ INPUT: generation_dist ───────┐
│ 7. output lined up by days     │ │ 8. total shedding of one       │ │ 9. infection_log: infectee     │
│    since date_exposed,         │ │    infection x 128 mL          │ │    infection day minus         │
│    averaged, / total (sum = 1) │ │    (ASSUMPTION: 128 mL)        │ │    infector infection day      │
│ = shedding_dist (no unit)      │ │ = load_per_case (gc/infection) │ │ = generation_dist (from day 1) │
└────────────────────────────────┘ └────────────────────────────────┘ └────────────────────────────────┘

ASSUMPTION = our choice | our arithmetic = computed from Covasim output | INPUT = goes to EpiSewer
```

## Flow: from shedding_hub output to the EpiSewer inputs

```
DATA INPUT FOR EPISEWER
┌─ COVASIM OUTPUT ─────┐    ┌─ ASSUMPTION ─────────┐    ┌─ COMPUTED ───────────┐    ┌─ ASSUMPTION ─────────┐    ┌─ INPUT TO EPISEWER ──┐
│ viral_shedding_hist  │    │ load_gc_per_day =    │    │ concentration =      │    │ measurement =        │    │ ww_data = 2 columns: │
│ (agents x days)      │    │ sum over the agents  │    │ load_gc_per_day /    │    │ concentration        │    │ col 1: concentration │
│ unit: gc per mL      │───▶│ in the site of       │───▶│ flow_L_per_day       │───▶│ x lab noise          │───▶│ col 2: flow          │
│ of stool             │    │ viral_shedding_hist  │    │ (gc/L, no noise)     │    │ (mean 1, CV 0.2)     │    │ Covasim gc/litre and │
│                      │    │ x 128 mL of stool    │    │                      │    │ drop days at 0       │    │ litres/day become    │
│                      │    │ (Ref: Rose 2015)     │    │                      │    │                      │    │ gc/mL and mL/day     │
└──────────────────────┘    └──────────────────────┘    └──────────────────────┘    └──────────────────────┘    └──────────────────────┘
                                                                    ▲
                                                                    │
                            ┌─ ASSUMPTION ─────────┐                │
                            │ flow_L_per_day =     │                │
                            │ agents in the site   │                │
                            │ x 270 L per person   │                │
                            │ (Ref: Point Loma)    │────────────────┘
                            └──────────────────────┘


OTHER INPUTS (from the same Covasim tables)
┌─ COMPUTED ───────────────────────────┐    ┌─ INPUT TO EPISEWER ──────────────┐
│ 7. shedding_dist = share of total    │    │ shedding_dist                    │
│    shedding on each day after        │───▶│ (no unit, sums to 1)             │
│    date_exposed (mean over agents of │    │                                  │
│    viral_shedding_hist)              │    │                                  │
└──────────────────────────────────────┘    └──────────────────────────────────┘
┌─ ASSUMPTION ─────────────────────────┐    ┌─ INPUT TO EPISEWER ──────────────┐
│ 8. load_per_case = total of          │    │ load_per_case                    │
│    viral_shedding_hist over the days │───▶│ (gene copies per infection)      │
│    of one infection                  │    │                                  │
│    x 128 mL stool (Ref: Rose 2015)   │    │                                  │
└──────────────────────────────────────┘    └──────────────────────────────────┘
┌─ COMPUTED ───────────────────────────┐    ┌─ INPUT TO EPISEWER ──────────────┐
│ 9. generation_dist = infection day   │    │ generation_dist                  │
│    of the infectee minus that of     │───▶│ (from day 1)                     │
│    the infector (from infection_log) │    │                                  │
└──────────────────────────────────────┘    └──────────────────────────────────┘

ASSUMPTION = our choice | COMPUTED = worked out from Covasim output | INPUT TO EPISEWER = goes into the R call
Ref: Rose et al. 2015, Crit Rev Environ Sci Technol 45:1827-1879 (median fecal wet mass 128 g/person/day; 1 g taken as 1 mL)
Ref: Point Loma = City of San Diego fact sheet and annual reports (136-175 million gallons/day, ~2.2 million residents = 235-300 L/person/day)
```

## Flow: from shedding_hub output to the EpiSewer inputs

```
DATA INPUT FOR EPISEWER
┌─ COVASIM OUTPUT ─────┐    ┌─ ASSUMPTION ─────────┐    ┌─ COMPUTED ───────────┐    ┌─ ASSUMPTION ─────────┐    ┌─ INPUT TO EPISEWER ──┐
│ Viral_shedding_hist  │    │ Load_gc_per_day =    │    │ Concentration =      │    │ Measurement =        │    │ Ww_data = 2 columns: │
│ (agents x days)      │    │ sum over the agents  │    │ load_gc_per_day /    │    │ concentration        │    │ Col 1: concentration │
│ unit: gc per mL      │───▶│ in the site of       │───▶│ flow_L_per_day       │───▶│ x lab noise          │──  col 2: flow          │
│ of stool             │    │ viral_shedding_hist  │    │ (gc/L, no noise)     │    │ (mean 1, CV 0.2)     │    │ Covasim: gc/litre    │
│                      │    │ x 128 mL of stool    │    │                      │    │ drop days at 0       │    │ litres/day become    │
│                      │    │ (Ref: Rose 2015)     │    │                      │    │                      │    │ Episewer: gc/mL and mL/day     
└──────────────────────┘    └──────────────────────┘    └──────────────────────┘    └──────────────────────┘    └──────────────────────┘
                                                                    ▲
                                                                    │
                            ┌─ ASSUMPTION ─────────┐                │
                            │ Flow_L_per_day =     │                │
                            │ agents in the site   │                │
                            │ x 270 L per person   │                │
                            │ (Ref: Point Loma)    │────────────────┘
                            └──────────────────────┘

OTHER INPUTS (from the same Covasim tables)
┌─ COMPUTED ───────────────────────────────┐    ┌─ ASSUMPTION ─────────────────────────────┐    ┌─ COMPUTED ───────────────────────────────┐
│ 7. shedding_dist = share of total        │    │ 8. load_per_case = total of              │    │ 9. generation_dist = infection day       │
│    shedding on each day after            │    │    viral_shedding_hist over the days     │    │    of the infectee minus that of         │
│    date_exposed (mean over agents of     │    │    of one infection                      │    │    the infector (from infection_log)     │
│    viral_shedding_hist)                  │    │    x 128 mL stool (Ref: Rose 2015)       │    │                                          │
└──────────────────────────────────────────┘    └──────────────────────────────────────────┘    └──────────────────────────────────────────┘
                      │                                               │                                               │
                      ▼                                               ▼                                               ▼
┌─ INPUT TO EPISEWER ──────────────────────┐    ┌─ INPUT TO EPISEWER ──────────────────────┐    ┌─ INPUT TO EPISEWER ──────────────────────┐
│ shedding_dist                            │    │ load_per_case                            │    │ generation_dist                          │
│ (no unit, sums to 1)                     │    │ (gene copies per infection)              │    │ (from day 1)                             │
└──────────────────────────────────────────┘    └──────────────────────────────────────────┘    └──────────────────────────────────────────┘

ASSUMPTION = our choice | COMPUTED = worked out from Covasim output | INPUT TO EPISEWER = goes into the R call
Ref: Rose et al. 2015, Crit Rev Environ Sci Technol 45:1827-1879 (median fecal wet mass 128 g/person/day; 1 g taken as 1 mL)
Ref: Point Loma = City of San Diego fact sheet and annual reports (136-175 million gallons/day, ~2.2 million residents = 235-300 L/person/day)
```